## Name: Asit Jain
## Roll No: M25DE1049
## Assignment 3 - MLBD

# Part 1: Content-Based Filtering
## Task 1: Implementing TF-IDF Based Recommendation

Build a content-based recommender system using TF-IDF on the MovieLens (ml-latest-small) dataset.

**Dataset:** https://grouplens.org/datasets/movielens/latest/ (1 MB - ml-latest-small)

In [1]:
# Install required libraries into the EXACT Python running this notebook
import subprocess, sys
subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'pandas', 'scikit-learn', '-q'])

import os
import zipfile
import urllib.request
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
print('All imports successful!')

All imports successful!


### Step 1: Download and Load the Dataset

In [2]:
# Download the MovieLens ml-latest-small dataset (1 MB)
url = "https://files.grouplens.org/datasets/movielens/ml-latest-small.zip"
zip_path = "ml-latest-small.zip"
extract_dir = "ml-latest-small"

if not os.path.exists(extract_dir):
    print("Downloading dataset...")
    urllib.request.urlretrieve(url, zip_path)
    with zipfile.ZipFile(zip_path, 'r') as z:
        z.extractall('.')
    print("Dataset downloaded and extracted.")
else:
    print("Dataset already exists.")

Dataset already exists.


In [3]:
# Load movies.csv — contains movieId, title, and genres
movies = pd.read_csv(os.path.join(extract_dir, 'movies.csv'))
print(f"Total movies: {len(movies)}")
movies.head(10)

Total movies: 9742


,movieId,title,genres
0,1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy
1,2,Jumanji (1995),Adventure|Children|Fantasy
2,3,Grumpier Old Men (1995),Comedy|Romance
3,4,Waiting to Exhale (1995),Comedy|Drama|Romance
4,5,Father of the Bride Part II (1995),Comedy
5,6,Heat (1995),Action|Crime|Thriller
6,7,Sabrina (1995),Comedy|Romance
7,8,Tom and Huck (1995),Adventure|Children
8,9,Sudden Death (1995),Action
9,10,GoldenEye (1995),Action|Adventure|Thriller


### Step 2: Extract and Prepare Genre Descriptions

The `genres` column contains pipe-separated genres (e.g. `Action|Adventure|Sci-Fi`).  
We replace `|` with spaces so TF-IDF can treat each genre as a separate term.

In [4]:
# Replace '|' with space and handle '(no genres listed)'
movies['genre_text'] = movies['genres'].str.replace('|', ' ', regex=False)
movies['genre_text'] = movies['genre_text'].replace('(no genres listed)', '')

movies[['title', 'genres', 'genre_text']].head(10)

,title,genres,genre_text
0,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy,Adventure Animation Children Comedy Fantasy
1,Jumanji (1995),Adventure|Children|Fantasy,Adventure Children Fantasy
2,Grumpier Old Men (1995),Comedy|Romance,Comedy Romance
3,Waiting to Exhale (1995),Comedy|Drama|Romance,Comedy Drama Romance
4,Father of the Bride Part II (1995),Comedy,Comedy
5,Heat (1995),Action|Crime|Thriller,Action Crime Thriller
6,Sabrina (1995),Comedy|Romance,Comedy Romance
7,Tom and Huck (1995),Adventure|Children,Adventure Children
8,Sudden Death (1995),Action,Action
9,GoldenEye (1995),Action|Adventure|Thriller,Action Adventure Thriller


### Step 3: Compute TF-IDF Vectors

In [5]:
# Build TF-IDF matrix from genre descriptions
tfidf = TfidfVectorizer(stop_words='english')
tfidf_matrix = tfidf.fit_transform(movies['genre_text'])

print(f"TF-IDF matrix shape: {tfidf_matrix.shape}")
print(f"Feature names (genres): {tfidf.get_feature_names_out()}")

TF-IDF matrix shape: (9742, 21)
Feature names (genres): ['action' 'adventure' 'animation' 'children' 'comedy' 'crime'
 'documentary' 'drama' 'fantasy' 'fi' 'film' 'horror' 'imax' 'musical'
 'mystery' 'noir' 'romance' 'sci' 'thriller' 'war' 'western']


### Step 4: Compute Cosine Similarity Between Items

In [6]:
# Compute pairwise cosine similarity
cosine_sim = cosine_similarity(tfidf_matrix, tfidf_matrix)

print(f"Cosine similarity matrix shape: {cosine_sim.shape}")

Cosine similarity matrix shape: (9742, 9742)


### Step 5: Implement the Recommendation Function

In [7]:
# Create a reverse mapping: movie title -> index
title_to_idx = pd.Series(movies.index, index=movies['title']).drop_duplicates()

def recommend_movies(title, top_n=5):
    """
    Given a movie title, return the top-N most similar movies
    based on cosine similarity of TF-IDF genre vectors.
    """
    # Check if the title exists
    if title not in title_to_idx:
        # Try partial match
        matches = movies[movies['title'].str.contains(title, case=False, na=False)]
        if matches.empty:
            print(f"Movie '{title}' not found in the dataset.")
            return None
        else:
            title = matches.iloc[0]['title']
            print(f"Closest match found: '{title}'")

    idx = title_to_idx[title]

    # Get similarity scores for this movie against all others
    sim_scores = list(enumerate(cosine_sim[idx]))

    # Sort by similarity (descending), skip the first one (itself)
    sim_scores = sorted(sim_scores, key=lambda x: x[1], reverse=True)[1:top_n+1]

    # Build result DataFrame
    results = pd.DataFrame({
        'Title': [movies.iloc[i]['title'] for i, _ in sim_scores],
        'Genres': [movies.iloc[i]['genres'] for i, _ in sim_scores],
        'Cosine Similarity': [round(score, 4) for _, score in sim_scores]
    })

    return results

### Step 6: Test with Sample Queries

In [8]:
# Query 1: Toy Story (1995)
print("="*70)
print("Recommendations for: Toy Story (1995)")
print("="*70)
recommend_movies('Toy Story (1995)', top_n=5)

Recommendations for: Toy Story (1995)


,Title,Genres,Cosine Similarity
0,Antz (1998),Adventure|Animation|Children|Comedy|Fantasy,1.0
1,Toy Story 2 (1999),Adventure|Animation|Children|Comedy|Fantasy,1.0
2,"Adventures of Rocky and Bullwinkle, The (2000)",Adventure|Animation|Children|Comedy|Fantasy,1.0
3,"Emperor's New Groove, The (2000)",Adventure|Animation|Children|Comedy|Fantasy,1.0
4,"Monsters, Inc. (2001)",Adventure|Animation|Children|Comedy|Fantasy,1.0


In [9]:
# Query 2: The Matrix (1999)
print("="*70)
print("Recommendations for: Matrix, The (1999)")
print("="*70)
recommend_movies('Matrix, The (1999)', top_n=5)

Recommendations for: Matrix, The (1999)


,Title,Genres,Cosine Similarity
0,Screamers (1995),Action|Sci-Fi|Thriller,1.0
1,Johnny Mnemonic (1995),Action|Sci-Fi|Thriller,1.0
2,Virtuosity (1995),Action|Sci-Fi|Thriller,1.0
3,Timecop (1994),Action|Sci-Fi|Thriller,1.0
4,Blade Runner (1982),Action|Sci-Fi|Thriller,1.0


In [10]:
# Query 3: Pulp Fiction (1994)
print("="*70)
print("Recommendations for: Pulp Fiction (1994)")
print("="*70)
recommend_movies('Pulp Fiction (1994)', top_n=5)

Recommendations for: Pulp Fiction (1994)


,Title,Genres,Cosine Similarity
0,Fargo (1996),Comedy|Crime|Drama|Thriller,1.0
1,Freeway (1996),Comedy|Crime|Drama|Thriller,1.0
2,Man Bites Dog (C'est arrivé près de chez vous)...,Comedy|Crime|Drama|Thriller,1.0
3,Beautiful Creatures (2000),Comedy|Crime|Drama|Thriller,1.0
4,Confessions of a Dangerous Mind (2002),Comedy|Crime|Drama|Thriller,1.0


In [11]:
# Query 4: Partial title search — just "Jurassic"
print("="*70)
print("Recommendations for: 'Jurassic' (partial match)")
print("="*70)
recommend_movies('Jurassic', top_n=5)

Recommendations for: 'Jurassic' (partial match)
Closest match found: 'Jurassic Park (1993)'


,Title,Genres,Cosine Similarity
0,Independence Day (a.k.a. ID4) (1996),Action|Adventure|Sci-Fi|Thriller,1.0
1,Escape from L.A. (1996),Action|Adventure|Sci-Fi|Thriller,1.0
2,"Abyss, The (1989)",Action|Adventure|Sci-Fi|Thriller,1.0
3,Escape from New York (1981),Action|Adventure|Sci-Fi|Thriller,1.0
4,Star Trek: First Contact (1996),Action|Adventure|Sci-Fi|Thriller,1.0


In [12]:
# Query 5: Forrest Gump (1994)
print("="*70)
print("Recommendations for: Forrest Gump (1994)")
print("="*70)
recommend_movies('Forrest Gump (1994)', top_n=5)

Recommendations for: Forrest Gump (1994)


,Title,Genres,Cosine Similarity
0,Life Is Beautiful (La Vita è bella) (1997),Comedy|Drama|Romance|War,1.0000
1,Train of Life (Train de vie) (1998),Comedy|Drama|Romance|War,1.0000
2,"Tiger and the Snow, The (La tigre e la neve) (...",Comedy|Drama|Romance|War,1.0000
3,I Served the King of England (Obsluhoval jsem ...,Comedy|Drama|Romance|War,1.0000
4,"Colonel Chabert, Le (1994)",Drama|Romance|War,0.9403
